<a href="https://colab.research.google.com/github/ShubhendraP/AgenticAI2026/blob/weekly-classes/Week7_Classroom_LangChain_Parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📋 Week 7 - Prompt Formatting & Response Parsing
### Professional Certificate Programme in Agentic AI and Applications

---

## 🎯 What You Will Learn
- How to design **structured prompts** that reliably control LLM output format
- How to use LangChain's **PromptTemplate** for reusable, dynamic prompts
- How to **parse LLM responses** into clean JSON / Python objects
- How to handle real-world parsing challenges (errors, missing fields, validation)

---

## 💡 Why This Matters

LLMs produce **free-form text**. But real systems need **structured data**:
- A database needs a dictionary, not a paragraph.
- An API needs JSON, not casual prose.
- A dashboard needs numbers, not sentences.

**Prompt Formatting** tells the LLM *how* to answer.  
**Response Parsing** turns that answer into usable structured data.

```
Free-form response (❌ hard to use):
  "The company OpenAI made over a billion dollars in revenue."

Parsed response (✅ machine-readable):
  {"company": "OpenAI", "revenue": "$1B+"}
```

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL'] = 'https://openai.vocareum.com/v1'

---
## 🔬 Part 1 — PromptTemplate: Reusable, Dynamic Prompts

**From Slides 25–29 and Slide 55 of the deck.**

A `PromptTemplate` is a prompt with **`{placeholders}`** that get filled with real values at runtime.  
Instead of writing a new prompt for every query, you write **one template** that handles thousands of inputs.

```
Template:  "Answer based on context below:\nContext: {context}\nQuestion: {question}\nAnswer:"

At runtime: context  = "Paris is the capital of France."
            question = "What is the capital of France?"

Filled prompt: "Answer based on context below:
               Context: Paris is the capital of France.
               Question: What is the capital of France?
               Answer:Paris
```

In [ ]:
!pip install langchain langchain-openai pydantic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.1 MB/s eta 0:00:00


In [ ]:
from langchain_core.prompts import PromptTemplate


template = '''Answer the following question basis the context below

Context: {context}
Question: {question}
Answer:'''


prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)


prompt = prompt_template.format(
    context="Paris is the capital of France.",
    question="What is the capital of France?"
)


print(prompt)




Answer the following question basis the context below 

Context: Paris is the capital of France.
Question: What is the capital of France?
Answer:


In [ ]:
## Parsing

---
## 🔬 Part 2 — JSON Response Parsing

**From Slides 33–34 of the deck.**

The most reliable way to parse LLM output is to **instruct it in the prompt** to return only JSON. Then use Python's `json.loads()` to convert the string into a usable dictionary.

### Why JSON?
- Can be inserted directly into SQL databases
- Parsed easily into Python dictionaries
- Formatted for API responses
- Compatible with frontend frameworks

In [ ]:
import json
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model = 'gpt-3.5-turbo', temperature=0)


json_template = PromptTemplate(
    input_variable = ['text'],
    template = '''Extract the company name and revene from the texts .
    Respond with JSON output only with the keys "company" and "revenue".
    Do not enter any extra fileds or text.
    Text: {text}
    JSON:'''
)


chain = json_template | llm | StrOutputParser()

# Example 1

text1 = 'OpenAI , is a company behind ChatGPT , It is said that the revenue of this company passes $1B  last year.'
result = chain.invoke({'text': text1})
print(result)


parsed = json.loads(result)
print(parsed)

## Parse the JSON string into a python dictionary






{
    "company": "OpenAI",
    "revenue": "$1B"
}
{'company': 'OpenAI', 'revenue': '$1B'}


In [ ]:
type(parsed)

dict

In [ ]:
# Test with multiple texts to show reusability
texts = [
    "Microsoft Azure generated approximately $87.9 billion in revenue for fiscal year 2023.",
    "Anthropic, the AI safety startup, raised funding and is growing rapidly with estimated revenues around $100M.",
    "Google's parent Alphabet reported total revenue of $307.4 billion for the full year 2023."
]


exctracted_data = []

for text in texts :
  result = chain.invoke({'text': text})
  parsed = json.loads(result)
  exctracted_data.append(parsed)

print(exctracted_data)


[{'company': 'Microsoft Azure', 'revenue': '$87.9 billion'}, {'company': 'Anthropic', 'revenue': '$100M'}, {'company': 'Alphabet', 'revenue': '$307.4 billion'}]


In [ ]:
# Langchain Output Parser

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field


In [ ]:
## Creating a Pydanctic Schema

class CompanyInfo(BaseModel):
  company:str =  Field(description="Name of the company")
  revenue:str = Field(description="Annual Revenue of the company")
  industry:str = Field(description="Industry sector in which company is operating")


# Create a JSON parser with this Schema

parser = JsonOutputParser(pydantic_object=CompanyInfo)

print(parser.get_format_instructions())


prompt = PromptTemplate(
    template = "Extract the company information from the text below Text : {text} \n {format_instructions} ",
    input_variables=['text'],
    partial_variables={'format_instructions': parser.get_format_instructions()}
)


llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=1)

chain = prompt | llm | parser

# Run it
result = chain.invoke({
    "text": "Nvidia, the leading chip maker for AI, reported record revenue of $60.9 billion in fiscal 2024, dominating the semiconductor industry."
})


print("\n✅ Parsed and validated output:")
print(result)
print(f"\n🏢 Company:  {result['company']}")
print(f"💰 Revenue:  {result['revenue']}")
print(f"🏭 Industry: {result['industry']}")

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


---
## 📌 Key Takeaways

| Concept | What it does |
|---|---|
| `PromptTemplate` | Reusable prompt with `{variable}` placeholders |
| `chain.invoke({...})` | Runs the chain with specific variable values |
| Structured prompts | Include format instructions (JSON, YAML, keys) IN the prompt |
| `json.loads()` | Converts a JSON string to a Python dictionary |
| `safe_parse_json()` | Custom parser with regex fallback + key validation |
| `JsonOutputParser` | LangChain's built-in parser — auto-generates format instructions |
| `Pydantic BaseModel` | Defines a strict schema with types and descriptions |

### 🏭 Production Workflow (from Slide 36)
```
1. Formatted Prompt  →  Tell LLM exactly what format to use
2. Structured Response  →  LLM returns JSON/YAML/Markdown
3. Parse & Validate  →  Extract data with error handling
4. System Integration  →  Feed into DB, API, dashboard
```

---
### ▶️ Next Demo
**Demo 3** covers **Tool Calling & Agents** — giving the LLM the ability to call external tools like calculators and search engines autonomously.